In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from retrieve import load_translation_memory, fuzzy_retrieval
import json, datetime

In [ ]:
load_dotenv()  # Load environment variables from .env file
translation_memory_path = "../data/tm/translation_memory.jsonl"
new_source = "Click Victoria"
target_language = "Polish"
tm = load_translation_memory(translation_memory_path)
client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

In [ ]:
def tm_retrieval(tm: list,new_source: str, top_n: int = 1):
    """Search the loaded TM and return the best match fields."""
    score, record = fuzzy_retrieval(tm, new_source, top_n=1)[0]
    return score, record["source"], record["target"]

In [ ]:
print(tm_retrieval(tm, new_source))

In [ ]:
PROMPTS = {
    "v1_only_different": """You are a professional video games translator that translates text from English to {target_language}.
Use the approved translation as a reference for translating the new source into {target_language}.
Replace with a {target_language} translation only those words that are different from the reference. Do not translate words that are the same as the reference.
You must maintain the same formatting, tags, and placeholders as in the source text. Do not add any commentary or explanation.""",
    
    "v2_base": """You are a professional video game translator working from English to {target_language}.
You repair fuzzy translation-memory matches.
Produce the full {target_language} translation of the NEW source.
Start from the approved translation and change only the parts that the source edit requires.
Keep the unchanged parts identical to the approved translation.
The output must be a complete, fluent {target_language} sentence. Never leave any part in English.
Preserve all placeholders (for example $student_hp), tags, and formatting exactly as they appear.
Output only the {target_language} translation, with no commentary.""",

    "v3_agreement": """You are a professional video game translator working from English to {target_language}.
You repair fuzzy translation-memory matches.
Produce the full {target_language} translation of the NEW source.
Start from the approved translation and change only the parts that the source edit requires.
Keep the unchanged parts identical to the approved translation.
If the source edit changes gender, number, or person, update every dependent word so the whole sentence agrees.
The output must be a complete, fluent {target_language} sentence. Never leave any part in English.
Preserve all placeholders (for example $student_hp), tags, and formatting exactly as they appear.
Output only the {target_language} translation, with no commentary.""",

    "v4_source_changes": """You are a professional video game translator working from English to {target_language}.
You repair fuzzy translation-memory matches.
Produce the full {target_language} translation of the NEW source.
Compare the reference source with the new source. Keep only the parts whose English words are unchanged.
Re-translate every part whose English word changed, including forms of address (for example man vs lady), 
and any words that depend on them (for example he vs she, his vs her, him vs her).
Reference source (English): Mighty human, you have a sword.
Approved (Polish): Potężny człowieku, masz miecz.
New source (English): Young elf, you have a sword.
Correct output: Młody elfie, masz miecz.
If the source edit changes gender, number, or person, update every dependent word so the whole sentence agrees.
The output must be a complete, fluent {target_language} sentence. Never leave any part in English.
Preserve all placeholders (for example $student_hp), tags, and formatting exactly as they appear.
Output only the {target_language} translation, with no commentary.""",
}

In [ ]:
def build_repair_messages(new_source: str, target_language: str, tm_source: str, tm_target: str, prompt_version: str) -> list:
    """Assemble the system + user messages for the repair call."""
    
    system = PROMPTS[prompt_version].format(target_language=target_language)

    user = f"""Reference source (English): {tm_source}
Approved translation ({target_language}): {tm_target}
New source (English): {new_source}"""

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

In [ ]:
print(build_repair_messages(new_source, target_language, tm_retrieval(tm, new_source)[1], tm_retrieval(tm, new_source)[2], "v4_source_changes"))

In [ ]:
def call_repair(messages: list, model: str, client: OpenAI) -> str:
    """Send the messages to the API and return the translation."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip()



In [ ]:

def log_run(path, prompt_version, model, new_source, tm_source, tm_target, output, fuzzy_score):
    """Logging function to record the details of each run."""
    rec = {
        "ts": datetime.datetime.now().isoformat(timespec="seconds"),
        "prompt_version": prompt_version,
        "model": model,
        "fuzzy_score": fuzzy_score,
        "new_source": new_source,
        "tm_source": tm_source,
        "tm_target": tm_target,
        "output": output,
    }
    with open(path, "a", encoding="utf-8") as f:      # append: log grows over time
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
def repair_segment(new_source, tm, target_language, prompt_version, model, client,
                   log_path="../eval/prompt_runs.jsonl"):
    """Orchestration: retrieve -> build messages -> call LLM -> log -> return."""
    score, tm_source, tm_target = tm_retrieval(tm, new_source)
    messages = build_repair_messages(new_source, target_language, tm_source, tm_target, prompt_version)
    output = call_repair(messages, model, client)
    log_run(log_path, prompt_version, model, new_source, tm_source, tm_target, output, score)
    return output

In [ ]:
print(repair_segment("Click Victoria", tm, "Polish", "v4_source_changes", "gpt-4o", client))

## Testing the model with new strings

In [ ]:
test1 = "Elvish Runemaster"
test2 = "Strong units do 1 more damage for every successful strike in melee combat.  Strength is a trait possessed only by Humans. The Humans are known for their reso;oamce, and their great facility with swords. Some, however, are gifted with natural talent that exceeds their brethren. These humans inflict an additional point of damage with sword strike."
test3 = "Young lady, you have $student_hp hitpoints and a javelin. I’m fairly sure you’ll succeed."

In [ ]:
versions = ["v1_only_different", "v2_base", "v3_agreement", "v4_source_changes"]
models = ["gpt-4o-mini", "gpt-4o"]
cases = [test1, test2, test3]

for version in versions:
    for model in models:
        for case in cases:
            repair_segment(case, tm, "Polish", version, model, client)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # show full cell text, no "..."
pd.set_option("display.width", None)          # use the full display width

df = pd.read_json("../eval/prompt_runs.jsonl", lines=True)
df.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
df.head(3).T

print(df)

## Smoke Test on New Data 
Strings for testing from pl_httt file.

In [ ]:
tm = load_translation_memory("../data/tm/translation_memory.jsonl")

In [ ]:
client = OpenAI()
model = os.getenv("OPENAI_MODEL", "gpt-4o")
target_language = "Polish"

new_test = "This place makes me feel uneasy, even with the humans retreating. We should leave now."

In [ ]:
from repair import repair_segment, PROMPTS

prompt_version = "v4_source_changes"
output, score = repair_segment(new_test, tm, target_language, prompt_version, model, client)
print(output)
print(score)